In [0]:
import os
print(os.getcwd())

In [0]:
# ==========================================
# 01_BRONZE NOTEBOOK
# Foundry-Style Metadata Logging
# ==========================================

import uuid
from pyspark.sql.functions import current_timestamp

# ==========================================
# 1. Capture Run Information
# ==========================================

# Generate unique run_id
run_id = str(uuid.uuid4())

# Detect if running as Job or Manual
try:
    context = dbutils.notebook.entry_point.getDbutils().notebook().getContext()
    job_id = context.jobId().get()
    job_name = context.tags().apply("jobName")
except:
    job_id = "manual_run"
    job_name = "manual_execution"

# Environment (change to prod when needed)
env = "dev"

# ==========================================
# 2. Capture Git Information (Databricks Repo)
# ==========================================

import subprocess

try:
    git_commit = subprocess.check_output(
        ["git", "rev-parse", "HEAD"],
        cwd="."
    ).decode().strip()

    git_branch = subprocess.check_output(
        ["git", "rev-parse", "--abbrev-ref", "HEAD"],
        cwd="."
    ).decode().strip()

    git_repo = subprocess.check_output(
        ["git", "config", "--get", "remote.origin.url"],
        cwd="."
    ).decode().strip()

except Exception as e:
    git_commit = "git_error"
    git_branch = "git_error"
    git_repo = "git_error"

print("Commit:", git_commit)
print("Branch:", git_branch)
print("Repo:", git_repo)
# ==========================================
# 4. Create Metadata Table (If Not Exists)
# ==========================================

spark.sql("""
CREATE OR REPLACE TABLE pipeline_runs (
    run_id STRING,
    job_id STRING,
    job_name STRING,
    env STRING,
    git_commit STRING,
    git_branch STRING,
    git_repo STRING,
    start_time TIMESTAMP,
    end_time TIMESTAMP,
    status STRING
)
USING DELTA
""")

# ==========================================
# 5. Insert RUNNING Status
# ==========================================

spark.sql(f"""
INSERT INTO pipeline_runs (
    run_id,
    job_id,
    job_name,
    env,
    git_commit,
    git_branch,
    git_repo,
    start_time,
    end_time,
    status
)
VALUES (
    '{run_id}',
    '{job_id}',
    '{job_name}',
    '{env}',
    '{git_commit}',
    '{git_branch}',
    '{git_repo}',
    current_timestamp(),
    NULL,
    'RUNNING'
)
""")

# ==========================================
# 6. Bronze Logic (Sample Transformation)
# ==========================================

# Example dummy bronze logic
data = [("A", 100), ("B", 200)]
df = spark.createDataFrame(data, ["category", "amount"])

df.write.mode("overwrite").saveAsTable("bronze_sales")

print("Bronze layer completed.")

# ==========================================
# 7. Update Status to SUCCESS
# ==========================================

spark.sql(f"""
UPDATE pipeline_runs
SET 
    status = 'SUCCESS',
    end_time = current_timestamp()
WHERE run_id = '{run_id}'
""")

print("Pipeline execution completed successfully.")